# 08 – Price and basic accommodation characteristics
## Question
Does a city price difference remain after accounting for basic listing characteristics?
One OLS model: `log(price) ~ city + room_type + log(accommodates) + bedrooms + log1p(minimum_nights)`.
City and room type are categorical, with Barcelona and entire home/apt as references. Use positive finite price and capacity, nonnegative bedrooms, minimum nights ≥ 1, and complete model variables. Report HC3 heteroskedasticity-robust 95% confidence intervals. This is descriptive association, not causal identification or a prediction exercise.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src.study_utils import load_spain, export_table, save_figure, SPAIN
from src.cities import CITIES

data = load_spain()
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
import statsmodels.formula.api as smf


In [2]:
columns = ['price', 'accommodates', 'bedrooms', 'minimum_nights']
valid = (np.isfinite(data[columns]).all(axis=1) & data.price.gt(0) & data.accommodates.gt(0)
         & data.bedrooms.ge(0) & data.minimum_nights.ge(1) & data.room_type.notna())
model_data = data.loc[valid].copy()
model_data['city'] = pd.Categorical(model_data.city, categories=SPAIN)
model_data['room_type'] = pd.Categorical(model_data.room_type,
    categories=['Entire home/apt', 'Private room', 'Hotel room', 'Shared room'])
model_data['log_price'] = np.log(model_data.price)
model_data['log_accommodates'] = np.log(model_data.accommodates)
model_data['log1p_minimum_nights'] = np.log1p(model_data.minimum_nights)
model = smf.ols('log_price ~ C(city) + C(room_type) + log_accommodates + bedrooms + log1p_minimum_nights',
                data=model_data).fit(cov_type='HC3')
X = model.model.exog
rank = np.linalg.matrix_rank(X)
assert rank == X.shape[1], 'Singular regression design'
assert np.isfinite(model.params).all() and np.isfinite(model.bse).all()
assert np.isfinite(model.conf_int()).all().all()
ci = model.conf_int()
coefficients = pd.DataFrame({'term': model.params.index, 'coefficient': model.params.values,
    'std_error_HC3':model.bse.values, 'ci_low':ci.iloc[:,0].values,
    'ci_high':ci.iloc[:,1].values, 'p_value':model.pvalues.values})
display(export_table(coefficients, 'study08_coefficients'))
fit = pd.DataFrame([{'n':int(model.nobs), 'r_squared':model.rsquared, 'adjusted_r_squared':model.rsquared_adj,
                     'rank':rank, 'columns':X.shape[1], 'condition_number':model.condition_number}])
display(export_table(fit, 'study08_fit'))
audit = data.groupby('city').agg(all_listings=('id','size'), missing_price=('price', lambda s:s.isna().sum()),
    missing_bedrooms=('bedrooms', lambda s:s.isna().sum()))
audit['complete_model'] = model_data.groupby('city', observed=True).size()
audit['excluded'] = audit.all_listings - audit.complete_model
display(export_table(audit, 'study08_filter_audit', index=True))


,term,coefficient,std_error_HC3,ci_low,ci_high,p_value
0,Intercept,5.021757,0.013618,4.995066,5.048447,0.000000e+00
1,C(city)[T.madrid],-0.275117,0.007458,-0.289734,-0.260500,6.685081e-298
2,C(room_type)[T.Private room],-0.407109,0.012977,-0.432543,-0.381676,4.819301e-216
3,C(room_type)[T.Hotel room],-0.089545,0.062229,-0.211511,0.032421,1.501583e-01
4,C(room_type)[T.Shared room],-1.234165,0.048847,-1.329903,-1.138426,7.591146e-141
5,log_accommodates,0.434316,0.022640,0.389944,0.478689,5.038025e-82
6,bedrooms,0.108024,0.012687,0.083158,0.132891,1.673539e-17
7,log1p_minimum_nights,-0.288168,0.003284,-0.294605,-0.281730,0.000000e+00


,n,r_squared,adjusted_r_squared,rank,columns,condition_number
0,26582,0.60494,0.604836,8,8,59.171962


,all_listings,missing_price,missing_bedrooms,complete_model,excluded
city,,,,,
barcelona,15293,1938,2562,11742,3551
madrid,22708,3536,5719,14840,7868


In [3]:
labels = {'C(city)[T.madrid]':'Madrid vs Barcelona', 'C(room_type)[T.Private room]':'Private vs entire',
    'C(room_type)[T.Hotel room]':'Hotel vs entire', 'C(room_type)[T.Shared room]':'Shared vs entire',
    'log_accommodates':'log(capacity)', 'bedrooms':'Bedrooms', 'log1p_minimum_nights':'log(1 + minimum nights)'}
selected = coefficients[coefficients.term.ne('Intercept')].copy()
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(selected.coefficient, np.arange(len(selected)),
    xerr=[selected.coefficient-selected.ci_low, selected.ci_high-selected.coefficient],
    fmt='o', color='#235789', capsize=4)
ax.set_yticks(np.arange(len(selected)), [labels.get(t,t) for t in selected.term])
ax.axvline(0,color='grey',linewidth=1)
ax.invert_yaxis()
ax.set(xlabel='OLS coefficient in log price (95% HC3 interval)', title='Price associations conditional on included characteristics')
fig.tight_layout(); save_figure(fig, 'study08_coefficients'); plt.show()
fig, ax = plt.subplots(figsize=(8, 5))
h = ax.hexbin(model.fittedvalues, model.resid, gridsize=45, mincnt=1, bins='log', cmap='Blues')
ax.axhline(0, color='#b33a3a', linewidth=1)
ax.set(xlabel='Fitted log price', ylabel='Residual (log price)', title='Residuals versus fitted values: all estimation rows')
fig.colorbar(h,ax=ax,label='Listings per bin (log colour scale)')
fig.tight_layout(); save_figure(fig, 'study08_residuals'); plt.show()
city_term = coefficients[coefficients.term.str.contains('madrid',case=False)].iloc[0]
print(f"Madrid conditional price association: {100*np.expm1(city_term.coefficient):.1f}% "
      f"(95% HC3 CI {100*np.expm1(city_term.ci_low):.1f}% to {100*np.expm1(city_term.ci_high):.1f}%).")


Madrid conditional price association: -24.1% (95% HC3 CI -25.2% to -22.9%).


/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11859/132073724.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save_figure(fig, 'study08_coefficients'); plt.show()
/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11859/132073724.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save_figure(fig, 'study08_residuals'); plt.show()


## Findings and limitations
The city coefficient compares expected log prices for otherwise equal included characteristics. `100 × (exp(beta) − 1)` expresses the corresponding multiplicative difference; it is not an arithmetic-mean price effect. R² refers to log prices. HC3 allows unequal residual variance but does not correct omitted variables, host clustering or spatial dependence. Exact location, quote dates, quoted stay length, amenities and quality are omitted. Complete-case selection, especially missing bedrooms, changes the analysis sample. The residual figure checks the simple model's fit without adding further models.


The fit uses 26,582 listings and has R² = 0.605. The Madrid coefficient is −0.275 (95% HC3 CI −0.290 to −0.261), corresponding to approximately 24.1% lower conditional prices (22.9–25.2% lower). The design has full rank: 8 of 8 columns.